## Model of "A 7-nm Compute-in-Memory SRAM Macro Supporting Multi-Bit Input, Weight and Output and Achieving 351 TOPS/W and 372.4 GOPS", JSSC 2021

Paper by Mahmut E. Sinangil, Burak Erbagci, Rawan Naous, Kerem Akarvardar, Dar Sun,
Win-San Khwa, Hung-Jen Liao, Yih Wang, and Jonathan Chang

### Description of The Macro

The macro uses a 64x64 SRAM array with every four columns connected by an analog adder.
This way, four side-by-side SRAM devices can store and compute with a 4b weight. A 4b
DAC provides 4b inputs, and a 4b ADC reads outputs from the array.

### Macro Level

- *Input Path*: 4b are encoded in the analog domain as a number of pulses from 0 to 15.
  To encode this input, a counter is used to count up to the input value generate the
  necessary number of pulses. Each row of the array is connected to a different counter.
  After passing through the counter, inputs (as pulse trains) pass through the row
  drivers.
- *Weight Path*: Weight drivers are used to rewrite weights in the array.
- *Output Path*: Column drivers read analog outputs from the array, after which outputs
  are passed through a 4b flash ADC to convert them to digital. There are 16 ADCs, one
  for each parallel output produced by the array.

Next, there are 16 columns in each macro. Inputs are spatially reused between columns
(*i.e,* each row wire connects to all columns), and each column processes separate
outputs and weights.

### Column Level

- *Input Path*: Each input is passed directly to a row in the column.
- *Weight Path*: A column bandwidth limiter sets the read and write bandwidth of each
  array column. Each weight is then passed to a row in the column.
- *Output Path*: Each column is composed of four wordlines that contain MAC results from
  four bits of weights. The outputs of these wordlines are weighted and summed with
  binary weighting capacitors. A column bandwidth limiter sets the read and write
  bandwidth of each array column.

Next, there are 16 rows in each column. Outputs are spatially reused between rows
(*i.e.,* accumulated on wires between rows), while each row processes separate inputs
and weights.

### Row Level

Inside each row, a cim_unit computes MAC operations.

- *Input Path*: A 4b input is provided to the cim_unit from the DAC.
- *Weight Path*: The cim_unit is composed of four SRAM devices that store a 4b weight.
- *Output Path*: Each cim_unit produces a 4b output.

Inside the CiM unit, 4x4x4 (4b input x 4b weight x 4b output) virtualized MAC units
compute the MAC operation.

In [ ]:
from _scripts import *

display_important_variables("sinangil_jssc_2021")
get_spec("sinangil_jssc_2021").arch

### Area Breakdown

This test replicates the results of Section IV paragraph 1 of the paper.

We show the area of the macro and its subcomponents. We report the area of the CIM
circuitry, the original macro, and the binary weighting capacitors.

The CiM circuitry consumes the majority of the area, due to the high area of ADC and DAC
circuitry.

In [ ]:
area = get_area_breakdown("sinangil_jssc_2021")

modeled = combine_areas(
    area,
    {
        "CiM Circuitry": ["ADC", "Counter"],
        "Original Macro": ["RowDrivers", "ColumnDrivers", "WeightDrivers", "CimUnit"],
        "Binary Weighting Capacitors": ["BinaryWeightingCapacitors"],
    },
)

reference = {
    "CiM Circuitry": 1596e-12,
    "Original Macro": 800e-12,
    "Binary Weighting Capacitors": 160e-12,
}

modeled_um2 = {k: v * 1e12 for k, v in modeled.items()}
reference_um2 = {k: v * 1e12 for k, v in reference.items()}

fig, ax = plt.subplots(figsize=(10, 5))
bar_comparison(
    {"Reference": reference_um2, "Modeled": modeled_um2},
    "Component",
    "Area (um²)",
    "Area Breakdown",
    ax,
)
plt.tight_layout()
plt.show()

print(f"{'Group':<32} {'Modeled':>12} {'Reference':>12} {'Ratio':>8}")
for k in ["CiM Circuitry", "Original Macro", "Binary Weighting Capacitors"]:
    m = modeled_um2.get(k, 0)
    r = reference_um2.get(k, 0)
    ratio = m / r if r > 0 else float("inf")
    print(f"{k:<32} {m:10.0f} um² {r:10.0f} um² {ratio:8.2f}x")

### Data-Value-Dependent Energy

This test replicates the results of Fig. 13(a) of the paper.

The energy consumed by the macro is data-value-dependent. In this test, we will plot the
average energy consumed by the macro as the average output value increases. We will vary
the output value from 0 to 15. For each, we set all input and weight values to be the
square root of the output value divided by a scaling factor. The scaling factor captures
the scaling of outputs before the ADC and the summation of results from many rows.

We see that the energy consumed by the macro increases as the average output value
increases. This is because:

- As the average input value increases, there is more switching on row wires to encode
  the larger input values, leading to higher energy.
- Similarly, as the average weight value increases, there is greater discharging of
  summing capacitors on each column, again leading to higher energy.

In [ ]:
output_scale_factor = 0.25
expected = {
    0: 3.35660091e-12,
    1: 4.562974203e-12,
    2: 5.033383915e-12,
    4: 5.443095599e-12,
    6: 5.837632777e-12,
    8: 6.285280728e-12,
    10: 6.657056146e-12,
    12: 7.04400607e-12,
    15: 7.795144158e-12,
}


def _run_sinangil_value(avg_out):
    spec = get_spec("sinangil_jssc_2021", add_dummy_main_memory=True)
    spec.arch.variables["average_input_value"] = output_scale_factor * (avg_out**0.5)
    spec.arch.variables["average_weight_value"] = output_scale_factor * (avg_out**0.5)
    spec.arch.variables["average_output_value"] = avg_out
    r = spec.map_workload_to_arch(print_progress=False)
    return avg_out, r.energy()


results = parallel([delayed(_run_sinangil_value)(x) for x in range(16)])

xs = [x for x, _ in results]
modeled_pj = [v * 1e12 for _, v in results]

fig, (ax_line, ax_bar) = plt.subplots(1, 2, figsize=(14, 5))

ref_xs = sorted(expected.keys())
ref_ys = [expected[k] * 1e12 for k in ref_xs]
ax_line.plot(ref_xs, ref_ys, "o-", label="reference")
ax_line.plot(xs, modeled_pj, "s-", label="modeled")
ax_line.set_xlabel("Average Output Value")
ax_line.set_ylabel("Energy (pJ per Full Array of MACs)")
ax_line.set_title("Output Value vs. Energy")
ax_line.set_xticks(xs)
ax_line.legend()
ax_line.grid(alpha=0.3)

bar_comparison(
    {
        "Reference": {f"out {k}": expected[k] * 1e12 for k in ref_xs},
        "Modeled": {f"out {k}": modeled_pj[xs.index(k)] for k in ref_xs},
    },
    "Average Output Value",
    "Energy (pJ)",
    "Reference vs. Modeled Energy",
    ax_bar,
)
plt.tight_layout()
plt.show()

print(f"{'avg_out':>8} {'Modeled':>12} {'Reference':>12} {'Ratio':>8}")
for k in ref_xs:
    m = modeled_pj[xs.index(k)]
    r = expected[k] * 1e12
    print(f"{k:>8} {m:10.3f} pJ {r:10.3f} pJ {m / r:7.2f}x")

### Energy Efficiency and Throughput

This test replicates the results of Table I in the paper. We show the area and energy
efficiency and throughput of the macro at 0.8V and 1V supply voltages and at varying
average output values. For output values, we use 0 (minimum, best-case), 6 (average),
and 15 (maximum, worst-case).

We see that, as the supply voltage increases, the macro gains throughput at the cost of
decreased energy efficiency. As the average output value increases, the energy
efficiency also decreases due to the data-value-dependent energy consumption of row and
column circuitry.

There is significant variation only for the case with 1V supply voltage and average
output value of 6. We attribute this to variation in measurements of the fabricated
chip, as all reference results except for this case follow consistent scaling trends
with supply voltage and average output value. Further tests on the fabricated chip may
be required to find the precise cause of this difference.

In [ ]:
configs = [(v, y) for v in [0.8, 1.0] for y in [15, 6, 0]]
reference_tops = [0.3724] * 3 + [0.4551] * 3
reference_tops_w = [262.3, 351.0, 610.5, 189.3, 321.0, 435.5]


def _run_sinangil_tops(v, y):
    spec = get_spec("sinangil_jssc_2021", add_dummy_main_memory=True)
    spec.variables.voltage = v
    spec.arch.variables["average_input_value"] = output_scale_factor * (y**0.5)
    spec.arch.variables["average_weight_value"] = output_scale_factor * (y**0.5)
    spec.arch.variables["average_output_value"] = y
    pc = spec.map_workload_to_arch(print_progress=False).per_compute()
    return v, y, 2 / pc.latency() / 1e12, 2 / pc.energy() / 1e12


out = parallel([delayed(_run_sinangil_tops)(v, y) for v, y in configs])
labels = [f"{v}V, out {y}" for v, y, _, _ in out]
model_tops = [t for _, _, t, _ in out]
model_tops_w = [tw for _, _, _, tw in out]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bar_comparison(
    {
        "Reference": dict(zip(labels, reference_tops)),
        "Modeled": dict(zip(labels, model_tops)),
    },
    "Voltage & Output Value",
    "Throughput (TOPS)",
    "Voltage & Output Value vs. Throughput",
    axes[0],
)
bar_comparison(
    {
        "Reference": dict(zip(labels, reference_tops_w)),
        "Modeled": dict(zip(labels, model_tops_w)),
    },
    "Voltage & Output Value",
    "Energy Efficiency (TOPS/W)",
    "Voltage & Output Value vs. Energy Efficiency",
    axes[1],
)
plt.tight_layout()
plt.show()

print(f"{'Config':<16} {'TOPS':>8} {'Ref TOPS':>10} {'TOPS/W':>10} {'Ref TOPS/W':>12}")
for lab, mt, rt, mw, rw in zip(
    labels, model_tops, reference_tops, model_tops_w, reference_tops_w
):
    print(f"{lab:<16} {mt:8.3f} {rt:10.3f} {mw:10.1f} {rw:12.1f}")

### Voltage Scaling

This test replicates the results of Fig. 13(b) of the paper.

We show the area and energy of the macro at supply voltages ranging from 0.65V to 1V,
testing the worst-case (inputs, weights, and outputs are all maximum values) scenario.
We see that the macro's energy consumption increases as the supply voltage increases.

When developing the model, we found that there is a energy scales more aggressively with
supply voltage in this result (Fig. 13(b) in the paper) than in the previous result
(Table I). This difference could have been due to many factors in the fabricated chip
measurement (temperature, input and output value distributions, clock frequency, etc.).
We chose to set up the model to match Table I, leading to a discrepancy in this result.

In [ ]:
def _run_sinangil_voltage(v):
    spec = get_spec("sinangil_jssc_2021", add_dummy_main_memory=True)
    spec.variables.voltage = v
    spec.arch.variables["average_input_value"] = output_scale_factor * (15**0.5)
    spec.arch.variables["average_weight_value"] = output_scale_factor * (15**0.5)
    spec.arch.variables["average_output_value"] = 15
    return v, float(spec.map_workload_to_arch(print_progress=False).energy())


voltages = [v / 100 for v in range(65, 105, 5)]
reference_pj = [5.109, 5.875, 6.797, 7.813, 9.031, 10.25, 11.75, 13.125]

results = parallel([delayed(_run_sinangil_voltage)(v) for v in voltages])
model_pj = [e * 1e12 for _, e in results]

fig, (ax_line, ax_bar) = plt.subplots(1, 2, figsize=(14, 5))
ax_line.plot(voltages, reference_pj, "o-", label="reference")
ax_line.plot(voltages, model_pj, "s-", label="modeled")
ax_line.set_xlabel("Voltage (V)")
ax_line.set_ylabel("Energy (pJ per Full Array of MACs)")
ax_line.set_title("Voltage vs. Energy")
ax_line.legend()
ax_line.grid(alpha=0.3)

labels = [f"{v}V" for v in voltages]
bar_comparison(
    {
        "Reference": dict(zip(labels, reference_pj)),
        "Modeled": dict(zip(labels, model_pj)),
    },
    "Voltage (V)",
    "Energy (pJ)",
    "Reference vs. Modeled Energy",
    ax_bar,
)
plt.tight_layout()
plt.show()

print(f"{'Voltage':>8} {'Modeled':>12} {'Reference':>12} {'Ratio':>8}")
for (v, e), ref in zip(results, reference_pj):
    m = e * 1e12
    print(f"{v:>7.2f}V {m:10.3f} pJ {ref:10.3f} pJ {m / ref:7.2f}x")

### Exploration of CiM unit width versus number of weight bits

This test explores the tradeoff between the width of the CiM unit (i.e., number of
weight bits that are stored and processed in one slice), the number of weight bits in
the workload, and the compute density of the macro.

In this test, we vary the CiM unit width while keeping the array size constant (*e.g.,*
when we double the CiM unit width, it doubles the bits per weight slice but halves the
number of columns). We then measure the throughput of the macro for different numbers of
bits per weight. When changing the CiM unit width, the binary weighting capacitors will
also be scaled to sum results from the bits within a weight slice.

We see that CiM units with more weight bits can increase compute density for a given
number of weight bits because they store more bits in each slice, require fewer columns
to store slices, and require less circuitry (mostly ADCs) to read outputs. However, CiM
units become underutilized when there are fewer bits per weight than they store.

Wider CiM units also lead to a larger-area chip due to the larger binary weighting
capacitors required to sum results from the bits within a weight slice. The size of the
binary weighting capacitors increases exponentially with the number of bits per weight.
For this reason, the eight-wide CiM unit had high area from the binary weighting
capacitors, and it never had the highest compute density.

In [ ]:
def _run_sinangil_explore(width_cells, weight_bits):
    spec = get_spec("sinangil_jssc_2021", add_dummy_main_memory=True)
    n_columns = 64 // width_cells
    spec.variables.cim_unit_width_cells = width_cells
    spec.variables.n_columns = n_columns
    spec.variables.supported_weight_bits = max(weight_bits, 1)
    spec.arch.variables["cim_unit_width_cells"] = width_cells
    spec.arch.variables["n_columns"] = n_columns
    cim_units_per_weight = max(1, (weight_bits + width_cells - 1) // width_cells)
    n_weights_per_row = max(1, n_columns // cim_units_per_weight)
    spec.workload.rank_sizes.update({"N": n_weights_per_row})
    for einsum in spec.workload.einsums:
        for ta in einsum.tensor_accesses:
            if ta.name == "weight":
                ta.bits_per_value = weight_bits
    r = spec.map_workload_to_arch(print_progress=False)
    ev = spec.calculate_component_costs()
    total_area = sum(float(v) for v in ev.arch.per_component_total_area.values())
    n_comp = float(r.n_computes())
    lat = float(r.latency())
    tops = 2 * n_comp / lat / 1e12 if lat > 0 else 0
    tops_mm2 = tops / (total_area * 1e6) if total_area > 0 else 0
    return width_cells, weight_bits, tops_mm2


widths = [1, 2, 4, 8]
weight_bits_range = list(range(1, 9))
explore = parallel(
    [delayed(_run_sinangil_explore)(w, b) for w in widths for b in weight_bits_range]
)

by_width = {w: [] for w in widths}
for w, b, td in explore:
    by_width[w].append((b, td))

fig, ax = plt.subplots(figsize=(8, 5))
for w in widths:
    pairs = sorted(by_width[w])
    bs = [p[0] for p in pairs]
    ds = [p[1] for p in pairs]
    ax.plot(bs, ds, "o-", label=str(w))
ax.set_xlabel("Weight Bits")
ax.set_ylabel("Compute Density (TOPS/mm²)")
ax.set_title("Weight Bits vs. Compute Density for Varied CiM Unit Widths")
ax.legend(title="cim_unit_width_cells")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"{'width_cells':>12} {'weight_bits':>12} {'TOPS/mm²':>10}")
for w in widths:
    for b, td in sorted(by_width[w]):
        print(f"{w:>12} {b:>12} {td:10.2f}")